# 第2回 実装課題：自習室FAQの学習前後比較

架空の自習室FAQ5項目を題材に、LoRA学習前後の回答を比較した。推論時にはFAQ本文と質問を入力する。

元ノートブック：`notebooks/lora_format_tuning.ipynb`。


## 0. 実行環境

Google ColabのTesla T4を使用。モデルのrevisionと主要ライブラリの版を固定した。


In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.14.1+cu130'}


## 1. 学習設定

20エポック、学習率0.0002、LoRAランク16、生成上限96トークン。モデル・revision・主要ライブラリの固定版は元ノートブックを引き継いだ。


In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/lora_implementation",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/lora_implementation",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

Mounted at /content/drive
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/lora_implementation",
  "drive_dir": "/content/drive/MyDrive/aie2026/lora_implementation"
}


## 2. 学習・検証・評価データ

学習14件、検証7件、評価21問（根拠あり15問・根拠なし6問）。根拠なしの質問はFAQに答えがない事項から用意した。

質問文はtrain / valid / testで重複しない。同じFAQ項目の言い換えは分割をまたぐ。testの正解は採点だけに使い、モデルの入力には渡さない。


In [4]:
PAYLOAD = json.loads(r"""{
  "task": "implementation",
  "system": "あなたは自習室 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に「確認が必要」と書き、needs_review を true にします。",
  "faq": {
    "hours": "自習室の利用時間は毎日 9:00〜21:00 です。",
    "reservation": "席の予約は利用日の 7 日前から、予約サイトで受け付けます。",
    "fee": "利用料金は 1 時間 300 円です。",
    "equipment": "Wi-Fi と電源を無料で利用できます。",
    "food": "ふた付きの飲み物は持ち込めますが、食事はできません。"
  },
  "train": [
    {
      "question": "自習室は何時から何時まで使えますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"自習室の利用時間は毎日 9:00〜21:00 です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "hours"
    },
    {
      "question": "自習室の営業時間を教えてください",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"自習室の利用時間は毎日 9:00〜21:00 です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "hours"
    },
    {
      "question": "席は何日前から予約できますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"席の予約は利用日の 7 日前から、予約サイトで受け付けます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "reservation"
    },
    {
      "question": "予約する方法と受付開始日を教えてください",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"席の予約は利用日の 7 日前から、予約サイトで受け付けます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "reservation"
    },
    {
      "question": "利用料金はいくらですか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"利用料金は 1 時間 300 円です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "fee"
    },
    {
      "question": "1 時間使うと何円かかりますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"利用料金は 1 時間 300 円です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "fee"
    },
    {
      "question": "Wi-Fi と電源は使えますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"Wi-Fi と電源を無料で利用できます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "equipment"
    },
    {
      "question": "ネット接続や充電に追加料金はかかりますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"Wi-Fi と電源を無料で利用できます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "equipment"
    },
    {
      "question": "飲食物は持ち込めますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"ふた付きの飲み物は持ち込めますが、食事はできません。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "food"
    },
    {
      "question": "自習室で食事をしてもいいですか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"ふた付きの飲み物は持ち込めますが、食事はできません。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "food"
    },
    {
      "question": "予約のキャンセル料はいくらですか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "学生割引はありますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "忘れ物は何日間保管されますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "プリンターの印刷料金はいくらですか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    }
  ],
  "valid": [
    {
      "question": "利用できる時間帯は？",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"自習室の利用時間は毎日 9:00〜21:00 です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "hours"
    },
    {
      "question": "席を予約したいときはどうしますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"席の予約は利用日の 7 日前から、予約サイトで受け付けます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "reservation"
    },
    {
      "question": "自習室の料金体系は？",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"利用料金は 1 時間 300 円です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "fee"
    },
    {
      "question": "無線 LAN とコンセントの利用条件は？",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"Wi-Fi と電源を無料で利用できます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "equipment"
    },
    {
      "question": "飲み物や食べ物についてのルールは？",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"ふた付きの飲み物は持ち込めますが、食事はできません。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "food"
    },
    {
      "question": "未成年だけで利用できますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "駐車場は何台分ありますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    }
  ],
  "test": [
    {
      "question": "何時に開いて何時に閉まりますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"自習室の利用時間は毎日 9:00〜21:00 です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "hours"
    },
    {
      "question": "自習室の開室時刻と閉室時刻を教えてください",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"自習室の利用時間は毎日 9:00〜21:00 です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "hours"
    },
    {
      "question": "何時から利用できて何時まで滞在できますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"自習室の利用時間は毎日 9:00〜21:00 です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "hours"
    },
    {
      "question": "いつからどこで予約できますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"席の予約は利用日の 7 日前から、予約サイトで受け付けます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "reservation"
    },
    {
      "question": "席の予約受付は利用日の何日前に始まりますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"席の予約は利用日の 7 日前から、予約サイトで受け付けます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "reservation"
    },
    {
      "question": "席の予約はどこで、いつから行えますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"席の予約は利用日の 7 日前から、予約サイトで受け付けます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "reservation"
    },
    {
      "question": "時間あたりの利用料を教えてください",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"利用料金は 1 時間 300 円です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "fee"
    },
    {
      "question": "自習室を 1 時間利用したときの金額は？",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"利用料金は 1 時間 300 円です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "fee"
    },
    {
      "question": "基本の時間料金を知りたいです",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"利用料金は 1 時間 300 円です。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "fee"
    },
    {
      "question": "Wi-Fi や電源は有料ですか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"Wi-Fi と電源を無料で利用できます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "equipment"
    },
    {
      "question": "Wi-Fi と電源を利用するための料金は？",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"Wi-Fi と電源を無料で利用できます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "equipment"
    },
    {
      "question": "ネットと電源は追加料金なしで使えますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"Wi-Fi と電源を無料で利用できます。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "equipment"
    },
    {
      "question": "飲食の決まりを教えてください",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"ふた付きの飲み物は持ち込めますが、食事はできません。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "food"
    },
    {
      "question": "ふたのある飲料と食事の持ち込みルールは？",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"ふた付きの飲み物は持ち込めますが、食事はできません。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "food"
    },
    {
      "question": "食べ物や飲み物を利用中に口にしてもよいですか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"ふた付きの飲み物は持ち込めますが、食事はできません。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "grounded",
      "term_id": "food"
    },
    {
      "question": "会議室も予約できますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "電話での予約はできますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "途中で外出して戻れますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "利用時間を当日に延長できますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "席は何席ありますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    },
    {
      "question": "満席のときにキャンセル待ちはできますか",
      "faq": "- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。",
      "target": "{\"answer\": \"この FAQ には記載がありません。運営に確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "ungrounded",
      "term_id": null
    }
  ],
  "gold": {
    "何時に開いて何時に閉まりますか": {
      "needs_review": false,
      "must_include": [
        "9:00",
        "21:00"
      ]
    },
    "自習室の開室時刻と閉室時刻を教えてください": {
      "needs_review": false,
      "must_include": [
        "9:00",
        "21:00"
      ]
    },
    "何時から利用できて何時まで滞在できますか": {
      "needs_review": false,
      "must_include": [
        "9:00",
        "21:00"
      ]
    },
    "いつからどこで予約できますか": {
      "needs_review": false,
      "must_include": [
        "7 日前",
        "予約サイト"
      ]
    },
    "席の予約受付は利用日の何日前に始まりますか": {
      "needs_review": false,
      "must_include": [
        "7 日前",
        "予約サイト"
      ]
    },
    "席の予約はどこで、いつから行えますか": {
      "needs_review": false,
      "must_include": [
        "7 日前",
        "予約サイト"
      ]
    },
    "時間あたりの利用料を教えてください": {
      "needs_review": false,
      "must_include": [
        "1 時間",
        "300 円"
      ]
    },
    "自習室を 1 時間利用したときの金額は？": {
      "needs_review": false,
      "must_include": [
        "1 時間",
        "300 円"
      ]
    },
    "基本の時間料金を知りたいです": {
      "needs_review": false,
      "must_include": [
        "1 時間",
        "300 円"
      ]
    },
    "Wi-Fi や電源は有料ですか": {
      "needs_review": false,
      "must_include": [
        "Wi-Fi",
        "電源",
        "無料"
      ]
    },
    "Wi-Fi と電源を利用するための料金は？": {
      "needs_review": false,
      "must_include": [
        "Wi-Fi",
        "電源",
        "無料"
      ]
    },
    "ネットと電源は追加料金なしで使えますか": {
      "needs_review": false,
      "must_include": [
        "Wi-Fi",
        "電源",
        "無料"
      ]
    },
    "飲食の決まりを教えてください": {
      "needs_review": false,
      "must_include": [
        "ふた付き",
        "飲み物",
        "食事はできません"
      ]
    },
    "ふたのある飲料と食事の持ち込みルールは？": {
      "needs_review": false,
      "must_include": [
        "ふた付き",
        "飲み物",
        "食事はできません"
      ]
    },
    "食べ物や飲み物を利用中に口にしてもよいですか": {
      "needs_review": false,
      "must_include": [
        "ふた付き",
        "飲み物",
        "食事はできません"
      ]
    },
    "会議室も予約できますか": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "電話での予約はできますか": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "途中で外出して戻れますか": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "利用時間を当日に延長できますか": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "席は何席ありますか": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "満席のときにキャンセル待ちはできますか": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    }
  },
  "source": {
    "path": "data/lora_homework/study_room.json",
    "sha256": "98353f7a6e57b5c8314c1e071c915b2a873484b1712b1f8cacdc4b3300b63b37",
    "note": "実装課題の題材案。すべて架空の自習室のルールであり、実在の施設の案内ではありません。"
  }
}""")

train, valid, test, gold = [PAYLOAD[k] for k in ("train", "valid", "test", "gold")]
FAQ = PAYLOAD["faq"]
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())
SYSTEM = PAYLOAD["system"]

def make_example(question, answer, needs_review):
    return {"question": question, "faq": FAQ_TEXT,
            "target": json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False),
            "needs_review": needs_review}

def build_messages(example):
    user = (f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
            if PAYLOAD["task"] == "implementation" else example["question"])
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

all_questions = [row["question"] for row in train + valid + test]
assert len(all_questions) == len(set(all_questions)), "質問が分割間で重複しています"
assert set(gold) == {row["question"] for row in test}
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("入力例:", build_messages(train[0]))
print("教師回答:", train[0]["target"])
for split, rows in [("train", train), ("valid", valid), ("test", test)]:
    records = [{"messages": build_messages(row) + [{"role": "assistant", "content": row["target"]}]} for row in rows]
    (WORK / f"{split}_messages.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in records), encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "source.json").write_text(json.dumps(PAYLOAD["source"], ensure_ascii=False, indent=2), encoding="utf-8")

train=14 valid=7 test=21 gold=21
入力例: [{'role': 'system', 'content': 'あなたは自習室 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {"answer": 文字列, "needs_review": 真偽値}。FAQ に記載が無いことは answer に「確認が必要」と書き、needs_review を true にします。'}, {'role': 'user', 'content': 'FAQ:\n- 自習室の利用時間は毎日 9:00〜21:00 です。\n- 席の予約は利用日の 7 日前から、予約サイトで受け付けます。\n- 利用料金は 1 時間 300 円です。\n- Wi-Fi と電源を無料で利用できます。\n- ふた付きの飲み物は持ち込めますが、食事はできません。\n\n質問: 自習室は何時から何時まで使えますか'}]
教師回答: {"answer": "自習室の利用時間は毎日 9:00〜21:00 です。", "needs_review": false}


186

## 3. モデル読み込み・学習前評価

形式はJSONのキーと型、内容は `needs_review` と必須語句で採点する。キーワード採点は正しい言い換えを落としたり、誤説明を見逃したりするため、回答原文も併せて確認する。


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# 教師回答が学習時に切り捨てられないか、tokenizerを使って確認する。
# Transformers 5では既定の戻り値が辞書なので、トークンIDのリストを明示して数える。
for split, rows in [("train", train), ("valid", valid)]:
    lengths = [len(tok.apply_chat_template(build_messages(row) + [{"role": "assistant", "content": row["target"]}],
                                         tokenize=True, add_generation_prompt=False,
                                         return_dict=False, truncation=False)) for row in rows]
    print(split, "最大トークン数:", max(lengths))
    assert max(lengths) <= CONFIG["max_seq_len"], "max_seq_len不足。設定を見直して0から実行してください"
target_lengths = [len(tok(row["target"], add_special_tokens=False)["input_ids"]) for row in test]
print("評価用の模範回答の最大トークン数:", max(target_lengths))
assert max(target_lengths) + 8 <= CONFIG["max_new_tokens"], "max_new_tokens不足。設定を見直して0から実行してください"

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01
train 最大トークン数: 225
valid 最大トークン数: 221
評価用の模範回答の最大トークン数: 30


In [6]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if not isinstance(obj, dict) or not isinstance(obj.get("answer"), str) or not isinstance(obj.get("needs_review"), bool): return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の全問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 21, "format_rate": 0.3333333333333333, "content_rate": 0.14285714285714285, "both_rate": 0.14285714285714285, "seconds": 22.1}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,何時に開いて何時に閉まりますか,False,"{""answer"": ""9:00〜21:00""}",×,missing_key,×
1,自習室の開室時刻と閉室時刻を教えてください,False,"{""answer"": ""確認が必要""}",×,missing_key,×
2,何時から利用できて何時まで滞在できますか,False,"{""answer"": ""9:00〜21:00""}",×,missing_key,×
3,いつからどこで予約できますか,False,"{""answer"": ""利用日の 7 日前から予約サイトから予約できます""}",×,missing_key,×
4,席の予約受付は利用日の何日前に始まりますか,False,"{""answer"": ""7"", ""needs_review"": ""false""}",×,wrong_type,×
5,席の予約はどこで、いつから行えますか,False,"{""answer"": ""予約サイト"", ""needs_review"": false}",○,,×
6,時間あたりの利用料を教えてください,False,"{""answer"": ""1 時間 300 円""}",×,missing_key,×
7,自習室を 1 時間利用したときの金額は？,False,"{""answer"": ""1 時間 300 円""}",×,missing_key,×
8,基本の時間料金を知りたいです,False,"{""answer"": ""1 時間 300 円""}",×,missing_key,×
9,Wi-Fi や電源は有料ですか,False,"{""answer"": ""確認が必要"", ""needs_review"": true}",○,,×


## 4. LoRA学習

SFTTrainerで学習し、学習時間・最後のログlossとそのステップ・学習全体の平均lossを記録する。最後のログlossと平均lossは区別する。


In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
train_output = trainer.train()
training_seconds = round(time.time() - t0, 1)
loss_logs = [row for row in trainer.state.log_history if "loss" in row]
last_loss_log = loss_logs[-1] if loss_logs else {}
print(f"学習時間: {training_seconds:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,1.649673
20,0.101853
30,0.024201
40,0.017353


学習時間: 205 秒 | ステップ数: 40 | エポック: 20.0


## 5. 学習前後の比較

同じ評価問題・生成条件で比較する。学習前に形式と内容の両方を満たし、学習後に不合格となった質問を退行とする。


In [8]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 21, "format_rate": 1.0, "content_rate": 0.9047619047619048, "both_rate": 0.9047619047619048, "seconds": 37.6}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,0.333333,0.142857,0.142857,22.1
after,21,1.000000,0.904762,0.904762,37.6


退行: ['途中で外出して戻れますか']


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,何時に開いて何時に閉まりますか,False,"{""answer"": ""9:00〜21:00""}",×/×,"{""answer"": ""自習室の利用時間は毎日 9:00〜21:00 です。"", ""needs_review"": false}",○/○
1,自習室の開室時刻と閉室時刻を教えてください,False,"{""answer"": ""確認が必要""}",×/×,"{""answer"": ""自習室の利用時間は毎日 9:00〜21:00 です。"", ""needs_review"": false}",○/○
2,何時から利用できて何時まで滞在できますか,False,"{""answer"": ""9:00〜21:00""}",×/×,"{""answer"": ""自習室の利用時間は毎日 9:00〜21:00 です。"", ""needs_review"": false}",○/○
3,いつからどこで予約できますか,False,"{""answer"": ""利用日の 7 日前から予約サイトから予約できます""}",×/×,"{""answer"": ""席の予約は利用日の 7 日前から、予約サイトで受け付けます。"", ""needs_review"": false}",○/○
4,席の予約受付は利用日の何日前に始まりますか,False,"{""answer"": ""7"", ""needs_review"": ""false""}",×/×,"{""answer"": ""席の予約は利用日の 7 日前から、予約サイトで受け付けます。"", ""needs_review"": false}",○/○
5,席の予約はどこで、いつから行えますか,False,"{""answer"": ""予約サイト"", ""needs_review"": false}",○/×,"{""answer"": ""席の予約は利用日の 7 日前から、予約サイトで受け付けます。"", ""needs_review"": false}",○/○
6,時間あたりの利用料を教えてください,False,"{""answer"": ""1 時間 300 円""}",×/×,"{""answer"": ""利用料金は 1 時間 300 円です。"", ""needs_review"": false}",○/○
7,自習室を 1 時間利用したときの金額は？,False,"{""answer"": ""1 時間 300 円""}",×/×,"{""answer"": ""利用料金は 1 時間 300 円です。"", ""needs_review"": false}",○/○
8,基本の時間料金を知りたいです,False,"{""answer"": ""1 時間 300 円""}",×/×,"{""answer"": ""利用料金は 1 時間 300 円です。"", ""needs_review"": false}",○/○
9,Wi-Fi や電源は有料ですか,False,"{""answer"": ""確認が必要"", ""needs_review"": true}",○/×,"{""answer"": ""Wi-Fi と電源を無料で利用できます。"", ""needs_review"": false}",○/○


## 6. 確認要求と根拠の確認

評価問題から根拠なし3問と根拠ありの対照1問を比較する。学習前の出力はadapterを無効化して取得する。「根拠あり」は正解ラベルに基づく。


In [9]:
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None:
        return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

probes = [row for row in test if row["needs_review"]][:3] + [next(row for row in test if not row["needs_review"])]
knowledge_rows = []
for ex in probes:
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": ex["question"], "根拠あり": not ex["needs_review"],
                           "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                           "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

,質問,根拠あり,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,会議室も予約できますか,False,needs_review,"{""answer"": ""確認が必要"", ""needs_review"": true}",needs_review,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}"
1,電話での予約はできますか,False,format_error,"{""answer"": ""確認が必要""}",needs_review,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}"
2,途中で外出して戻れますか,False,needs_review,"{""answer"": ""確認が必要"", ""needs_review"": true}",asserted,"{""answer"": ""ふた付きの飲み物は持ち込めますが、食事はできません。"", ""needs_review"": false}"
3,何時に開いて何時に閉まりますか,True,format_error,"{""answer"": ""9:00〜21:00""}",asserted,"{""answer"": ""自習室の利用時間は毎日 9:00〜21:00 です。"", ""needs_review"": false}"


## 7. 結果保存

設定、学習指標、質問別の評価結果、区分別の比較、データの出典をJSON・CSVに保存する。


In [10]:
group_rows = []
evaluations = [("before", before_rows), ("after", after_rows)]
if PAYLOAD["task"] == "glossary":
    evaluations.append(("search", search_rows))
for label, rows in evaluations:
    indexed = {row["question"]: row for row in rows}
    for group in sorted({ex["group"] for ex in test}):
        subset = [indexed[ex["question"]] for ex in test if ex["group"] == group]
        group_rows.append({"label": label, "group": group, "n": len(subset),
                           "format_rate": sum(row["format_ok"] for row in subset) / len(subset),
                           "content_rate": sum(row["content_ok"] for row in subset) / len(subset)})
group_comparison = pd.DataFrame(group_rows)
display(group_comparison)
group_comparison.to_csv(WORK / "group_comparison.csv", index=False, encoding="utf-8-sig")

result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "task": PAYLOAD["task"],
    "source": PAYLOAD["source"],
    "training": {"seconds": training_seconds, "final_logged_loss": last_loss_log.get("loss"),
                 "last_logged_step": last_loss_log.get("step"), "steps": trainer.state.global_step,
                 "mean_training_loss": train_output.training_loss},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
result["groups"] = group_rows
if PAYLOAD["task"] == "glossary":
    result["search"] = search_summary
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))

if USE_DRIVE:
    import shutil
    for path in WORK.iterdir():
        if path.is_file():
            shutil.copy(path, Path(CONFIG["drive_dir"]) / path.name)
    print("Driveに保存しました:", CONFIG["drive_dir"])
print("保存先:", WORK)

,label,group,n,format_rate,content_rate
0,before,grounded,15,0.266667,0.000000
1,before,ungrounded,6,0.500000,0.500000
2,after,grounded,15,1.000000,1.000000
3,after,ungrounded,6,1.000000,0.666667


{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.14.1+cu130"
  },
  "task": "implementation",
  "source": {
    "path": "data/lora_homework/study_room.json",
    "sha256": "98353f7a6e57b5c8314c1e071c915b2a873484b1712b1f8cacdc4b3300b63b37",
    "note": "実装課題の題材案。すべて架空の自習室のルールであり、実在の施設の案内ではありません。"
  },
  "training": {
    "seconds": 205.3,
    "final_logged_loss": 0.01735316962003708,
    "last_logged_step": 40,
    "steps": 40,
    "mean_training_loss": 0.44826995618641374
  },
  "method": "LoRA（量子化なし）",
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 0.3333333333333333,
    "co